# Man vs Machine on Oslo Bors - Machine Learning



In [16]:
# 1. CORE LIBRARIES
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 2. MACHINE LEARNING LIBRARIES
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score


### Data setup

In [17]:
# 3. LOAD THE CLEAN DATA
hist_df = pd.read_csv('../Data/investable_universe.csv')
modern_df = pd.read_csv('../Data/modern_equity_dataset.csv')

# Ensure datetime format for chronological splitting
hist_df['TradeDate'] = pd.to_datetime(hist_df['TradeDate'])
modern_df['TradeDate'] = pd.to_datetime(modern_df['TradeDate'])


In [18]:
# -------------------------------------------------------------------
# 4. DEFINE THE CHRONOLOGICAL SPLITS
# -------------------------------------------------------------------
# We use the exact same Validation and Test windows for both datasets
VAL_START = '2015-01-01'
TEST_START = '2018-01-01'

# Split Historical Data (1980 - Present)
hist_train = hist_df[hist_df['TradeDate'] < VAL_START].copy()
hist_val   = hist_df[(hist_df['TradeDate'] >= VAL_START) & (hist_df['TradeDate'] < TEST_START)].copy()
hist_test  = hist_df[hist_df['TradeDate'] >= TEST_START].copy()

# Split Modern Data (2010 - Present)
modern_train = modern_df[modern_df['TradeDate'] < VAL_START].copy()
modern_val   = modern_df[(modern_df['TradeDate'] >= VAL_START) & (modern_df['TradeDate'] < TEST_START)].copy()
modern_test  = modern_df[modern_df['TradeDate'] >= TEST_START].copy()

print(f"--- HISTORICAL TRACK ---")
print(f"Train: {hist_train.shape[0]} rows | Val: {hist_val.shape[0]} rows | Test: {hist_test.shape[0]} rows")

print(f"\n--- MODERN TRACK (With Best Ideas) ---")
print(f"Train: {modern_train.shape[0]} rows | Val: {modern_val.shape[0]} rows | Test: {modern_test.shape[0]} rows")

--- HISTORICAL TRACK ---
Train: 56291 rows | Val: 6406 rows | Test: 6191 rows

--- MODERN TRACK (With Best Ideas) ---
Train: 11373 rows | Val: 6406 rows | Test: 6191 rows


In [19]:
# -------------------------------------------------------------------
# 5. DEFINE FEATURES AND TARGETS
# -------------------------------------------------------------------
# Base features used by BOTH models
base_features = [
    'log_size_z', 'turnover_z', 
    'rev_1m_z', 'mom_6m_z', 'mom_12m_z', 
    'vol_3m_z', 'vol_6m_z'
]

# Smart Money features added ONLY to the Modern track
smart_money_features = [
    'BI_Avg_Weight_z', 
    'BI_Fund_Count_z', 
    'BI_Top10_Count_z'
]

modern_features = base_features + smart_money_features
target_col = 'Target_z'

# 6. SEPARATE INTO X AND y MATRICES
# Historical Track
X_train_hist, y_train_hist = hist_train[base_features], hist_train[target_col]
X_val_hist, y_val_hist     = hist_val[base_features], hist_val[target_col]
X_test_hist, y_test_hist   = hist_test[base_features], hist_test[target_col]

# Modern Track
X_train_mod, y_train_mod = modern_train[modern_features], modern_train[target_col]
X_val_mod, y_val_mod     = modern_val[modern_features], modern_val[target_col]
X_test_mod, y_test_mod   = modern_test[modern_features], modern_test[target_col]

print("X and y matrices successfully created for both tracks!")

X and y matrices successfully created for both tracks!


### Historical ML pipeline

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# -------------------------------------------------------------------
# 1. SETUP THE CHRONOLOGICAL "TIME MACHINE" SPLIT
# -------------------------------------------------------------------
# Combine Train and Validation sets
X_train_val_hist = pd.concat([X_train_hist, X_val_hist])
y_train_val_hist = pd.concat([y_train_hist, y_val_hist])

# Assign -1 to training rows (Train Only) and 0 to validation rows (Validate Only)
train_indices = np.full(len(X_train_hist), -1)
val_indices = np.zeros(len(X_val_hist))
split_array = np.concatenate([train_indices, val_indices])

# Create the Time-Series Splitter
time_split = PredefinedSplit(test_fold=split_array)


In [26]:
# -------------------------------------------------------------------
# 2. DEFINE THE MODELS AND HYPERPARAMETER GRIDS
# -------------------------------------------------------------------
# We restrict tree depth aggressively to prevent overfitting financial noise!
model_pipelines = {
    'Ridge': {
        'estimator': Ridge(random_state=123),
        'param_grid': {
            'alpha': [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
        }
    },
    'RandomForest': {
        'estimator': RandomForestRegressor(random_state=123, n_jobs=-1),
        'param_grid': {
            'max_depth': [5, 7, 10, 15],             # Expanded deeper
            'min_samples_leaf': [20, 50, 100],       # Allowed smaller leaves
            'n_estimators': [100, 200]
        }
    },
    'XGBoost': {
        'estimator': xgb.XGBRegressor(random_state=123, n_jobs=-1, objective='reg:squarederror'),
        'param_grid': {
            'max_depth': [2, 3, 5, 7, 10],           # Expanded deeper
            'learning_rate': [0.01, 0.05, 0.1],      # Added standard learning rate
            'subsample': [0.6, 0.8, 1.0],            # Added more subsample options
            'n_estimators': [100, 200]
        }
    }
}

In [27]:
# -------------------------------------------------------------------
# 3. INITIALIZE RESULTS DATAFRAME
# -------------------------------------------------------------------
# This dataframe will hold the real targets and all model predictions
hist_test_results = hist_test[['TradeDate', 'ISIN', 'Target_raw', 'Target_z']].copy()

# -------------------------------------------------------------------
# 4. EXECUTE THE GRID SEARCH LOOP
# -------------------------------------------------------------------
print("Starting Multi-Model GridSearchCV (This may take a few minutes)...\n")

for model_name, config in model_pipelines.items():
    print(f"--- Tuning {model_name} ---")
    
    # Setup GridSearch with our PredefinedSplit
    grid_search = GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['param_grid'],
        cv=time_split,
        scoring='neg_mean_squared_error',
        refit=True, # Automatically retrains on Train + Val with the best params
        verbose=1   # Prints progress
    )
    
    # Train and tune
    grid_search.fit(X_train_val_hist, y_train_val_hist)
    
    # Print the winning parameters
    best_params = grid_search.best_params_
    best_mse = -grid_search.best_score_ # Convert back to positive MSE
    print(f"Best Params: {best_params}")
    print(f"Validation MSE: {best_mse:.5f}\n")
    
    # Predict on the unseen Test Set (Simulation data)
    test_preds = grid_search.predict(X_test_hist)
    
    # Save the predictions into our results dataframe
    hist_test_results[f'{model_name}_Pred_z'] = test_preds


Starting Multi-Model GridSearchCV (This may take a few minutes)...

--- Tuning Ridge ---
Fitting 1 folds for each of 10 candidates, totalling 10 fits
Best Params: {'alpha': 1e-06}
Validation MSE: 0.98650

--- Tuning RandomForest ---
Fitting 1 folds for each of 24 candidates, totalling 24 fits
Best Params: {'max_depth': 10, 'min_samples_leaf': 20, 'n_estimators': 100}
Validation MSE: 0.98557

--- Tuning XGBoost ---
Fitting 1 folds for each of 90 candidates, totalling 90 fits
Best Params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.6}
Validation MSE: 0.98576



In [28]:
# -------------------------------------------------------------------
# 5. CREATE THE ENSEMBLE PREDICTION
# -------------------------------------------------------------------
# Calculate the simple average of all three models
hist_test_results['Ensemble_Pred_z'] = hist_test_results[
    ['Ridge_Pred_z', 'RandomForest_Pred_z', 'XGBoost_Pred_z']
].mean(axis=1)

print("All models successfully trained, tuned, and ensembled!")
print(hist_test_results.head())

All models successfully trained, tuned, and ensembled!
    TradeDate          ISIN  Target_raw  Target_z  Ridge_Pred_z  \
31 2018-01-31  AU0000057408    0.000000 -0.079296     -0.197935   
32 2018-02-28  AU0000057408   -0.011494  0.198976     -0.199512   
33 2018-03-31  AU0000057408    0.674419  3.769681     -0.140021   
34 2018-04-30  AU0000057408   -0.076389 -0.836654     -0.255447   
35 2018-05-31  AU0000057408   -0.090226 -0.760221     -0.160965   

    RandomForest_Pred_z  XGBoost_Pred_z  Ensemble_Pred_z  
31            -0.098640       -0.044922        -0.113833  
32            -0.092732       -0.098694        -0.130312  
33            -0.014429       -0.068914        -0.074455  
34            -0.543427       -0.609165        -0.469346  
35            -0.331967       -0.186204        -0.226379  


### Modern ML pipeline

In [29]:
# -------------------------------------------------------------------
# 1. SETUP THE CHRONOLOGICAL "TIME MACHINE" SPLIT
# -------------------------------------------------------------------
# Combine Train and Validation sets
X_train_val_mod = pd.concat([X_train_mod, X_val_mod])
y_train_val_mod = pd.concat([y_train_mod, y_val_mod])

# Assign -1 to training rows (Train Only) and 0 to validation rows (Validate Only)
train_indices = np.full(len(X_train_mod), -1)
val_indices = np.zeros(len(X_val_mod))
split_array = np.concatenate([train_indices, val_indices])

# Create the Time-Series Splitter
time_split = PredefinedSplit(test_fold=split_array)


In [30]:
# -------------------------------------------------------------------
# 2. DEFINE THE MODELS AND HYPERPARAMETER GRIDS
# -------------------------------------------------------------------
# We restrict tree depth aggressively to prevent overfitting financial noise!
model_pipelines = {
    'Ridge': {
        'estimator': Ridge(random_state=123),
        'param_grid': {
            'alpha': [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
        }
    },
    'RandomForest': {
        'estimator': RandomForestRegressor(random_state=123, n_jobs=-1),
        'param_grid': {
            'max_depth': [5, 7, 10, 15],             # Expanded deeper
            'min_samples_leaf': [20, 50, 100],       # Allowed smaller leaves
            'n_estimators': [100, 200]
        }
    },
    'XGBoost': {
        'estimator': xgb.XGBRegressor(random_state=123, n_jobs=-1, objective='reg:squarederror'),
        'param_grid': {
            'max_depth': [2, 3, 5, 7, 10],           # Expanded deeper
            'learning_rate': [0.01, 0.05, 0.1],      # Added standard learning rate
            'subsample': [0.6, 0.8, 1.0],            # Added more subsample options
            'n_estimators': [100, 200]
        }
    }
}

In [31]:
# -------------------------------------------------------------------
# 3. INITIALIZE RESULTS DATAFRAME
# -------------------------------------------------------------------
# This dataframe will hold the real targets and all model predictions
mod_test_results = modern_test[['TradeDate', 'ISIN', 'Target_raw', 'Target_z']].copy()

# -------------------------------------------------------------------
# 4. EXECUTE THE GRID SEARCH LOOP
# -------------------------------------------------------------------
print("Starting Multi-Model GridSearchCV (This may take a few minutes)...\n")

for model_name, config in model_pipelines.items():
    print(f"--- Tuning {model_name} ---")
    
    # Setup GridSearch with our PredefinedSplit
    grid_search = GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['param_grid'],
        cv=time_split,
        scoring='neg_mean_squared_error',
        refit=True, # Automatically retrains on Train + Val with the best params
        verbose=1   # Prints progress
    )
    
    # Train and tune
    grid_search.fit(X_train_val_mod, y_train_val_mod)
    
    # Print the winning parameters
    best_params = grid_search.best_params_
    best_mse = -grid_search.best_score_ # Convert back to positive MSE
    print(f"Best Params: {best_params}")
    print(f"Validation MSE: {best_mse:.5f}\n")
    
    # Predict on the unseen Test Set (Simulation data)
    test_preds = grid_search.predict(X_test_mod)
    
    # Save the predictions into our results dataframe
    mod_test_results[f'{model_name}_Pred_z'] = test_preds


Starting Multi-Model GridSearchCV (This may take a few minutes)...

--- Tuning Ridge ---
Fitting 1 folds for each of 10 candidates, totalling 10 fits
Best Params: {'alpha': 100.0}
Validation MSE: 0.98595

--- Tuning RandomForest ---
Fitting 1 folds for each of 24 candidates, totalling 24 fits
Best Params: {'max_depth': 10, 'min_samples_leaf': 100, 'n_estimators': 200}
Validation MSE: 0.98365

--- Tuning XGBoost ---
Fitting 1 folds for each of 90 candidates, totalling 90 fits
Best Params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100, 'subsample': 0.6}
Validation MSE: 0.98679



In [32]:
# -------------------------------------------------------------------
# 5. CREATE THE ENSEMBLE PREDICTION
# -------------------------------------------------------------------
# Calculate the simple average of all three models
mod_test_results['Ensemble_Pred_z'] = mod_test_results[
    ['Ridge_Pred_z', 'RandomForest_Pred_z', 'XGBoost_Pred_z']
].mean(axis=1)

print("All models successfully trained, tuned, and ensembled!")
print(mod_test_results.head())

All models successfully trained, tuned, and ensembled!
    TradeDate          ISIN  Target_raw  Target_z  Ridge_Pred_z  \
31 2018-01-31  AU0000057408    0.000000 -0.079296     -0.096381   
32 2018-02-28  AU0000057408   -0.011494  0.198976     -0.099899   
33 2018-03-31  AU0000057408    0.674419  3.769681     -0.077091   
34 2018-04-30  AU0000057408   -0.076389 -0.836654     -0.382654   
35 2018-05-31  AU0000057408   -0.090226 -0.760221     -0.251019   

    RandomForest_Pred_z  XGBoost_Pred_z  Ensemble_Pred_z  
31             0.018866        0.126124         0.016203  
32             0.088011        0.223457         0.070523  
33             0.061021        0.106462         0.030131  
34            -0.313467       -0.339086        -0.345069  
35            -0.253845       -0.106312        -0.203725  


### Neural Network pipelines


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("--- Tuning Neural Network (MLP) ---")

# 1. Create a Pipeline that scales the data right before feeding it to the NN
mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(random_state=123, max_iter=500, early_stopping=True, validation_fraction=0.1))
])

# 2. Define the Neural Network Grid
# Note: Because it's in a pipeline, we prefix the parameters with 'mlp__'
mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(8,), (16, 8), (32, 16)],  # Try a tiny network, a small one, and a medium one
    'mlp__activation': ['relu', 'tanh'],                   # How neurons activate
    'mlp__alpha': [0.1, 1.0, 10.0],                        # L2 Regularization (Weight Decay)
    'mlp__learning_rate_init': [0.001, 0.01]               # How fast the network updates weights
}

# 3. Setup GridSearchCV
mlp_grid_search = GridSearchCV(
    estimator=mlp_pipeline,
    param_grid=mlp_param_grid,
    cv=time_split,
    scoring='neg_mean_squared_error',
    refit=True,
    verbose=1,
    n_jobs=-1
)

# 4. Train and Tune
mlp_grid_search.fit(X_train_val_hist, y_train_val_hist)

# 5. Extract results
best_mlp_params = mlp_grid_search.best_params_
best_mlp_mse = -mlp_grid_search.best_score_
print(f"Best Params: {best_mlp_params}")
print(f"Validation MSE: {best_mlp_mse:.5f}\n")

# 6. Predict on Test Set
mlp_test_preds = mlp_grid_search.predict(X_test_hist)
hist_test_results['MLP_Pred_z'] = mlp_test_preds

# 7. Update the Ensemble to include all 4 models!
hist_test_results['Ensemble_Pred_z'] = hist_test_results[
    ['Ridge_Pred_z', 'RandomForest_Pred_z', 'XGBoost_Pred_z', 'MLP_Pred_z']
].mean(axis=1)

print("Neural Network added to the Test Results and Ensemble updated!")